In [1]:
from IPython.display import display, Markdown
from qdrant_client import QdrantClient
from qdrant_client.models import (
    PointStruct,
    Distance,
    VectorParams,
    SparseVector,
    models,
)
from fastembed import SparseTextEmbedding

import numpy as np
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_ar

/home/ismail/Documents/projects/python/ML/Summer/projects/islam/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Qdrant

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1"
    ,model_kwargs={"dtype": "float16"},
                            )

In [3]:
client = QdrantClient(url='http://localhost:6333')
# sparse_model = "qdrant/bm25"

In [ ]:
# if client.collection_exists('sunnah'):
#     client.delete_collection('sunnah')

# client.create_collection(
#     'sunnah',
#     vectors_config={
#         "dense": models.VectorParams(
#             size=768,
#             distance=models.Distance.COSINE,
#         ),
#     },
#     sparse_vectors_config={
#         "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
#     }
# )


True

# Sunnah

In [31]:
from datasets import Dataset,load_dataset
ds = load_dataset("SHK4K/athar_books", split="train")

In [ ]:
df = ds.to_pandas()

In [6]:
for book in df['book_id'].unique():
    chunks_num = np.arange(1,len(df[df['book_id']==book])+1)
    df.loc[df['book_id']==book,'chunk_num'] = chunks_num
    
ds = Dataset.from_pandas(df)
del df

In [7]:
def clean_hf(batch):
    return {'body_clean': [normalize_alef_ar(dediac_ar(text)) for text in batch['body']]}

ds = ds.map(clean_hf,num_proc=4,batched=True)

Map (num_proc=4):   0%|          | 0/110984 [00:00<?, ? examples/s]

Map (num_proc=4): 100%|██████████| 110984/110984 [00:08<00:00, 13622.18 examples/s]


In [8]:
embs = torch.load('tensors/sunnah.pt', weights_only=False).tolist()

In [9]:
ds.column_names

['book_id',
 'book_category',
 'book_type',
 'book_date',
 'book_name',
 'category_name',
 'all_authors',
 'main_author_death',
 'page_count',
 'headers',
 'body',
 'athar_num',
 'page',
 '__index_level_0__',
 'chunk_num',
 'body_clean']

In [10]:
def book_meta(row):
    return {
        'book_id': int(row['book_id']),
        'book_name':row['book_name'],
        'bood_data':row['book_date'],
        'category_name': row['category_name'],
        'all_authors': row['all_authors'],
        'headers': row['headers'],
        'page': row['page'],
        'athar_number': row['athar_num'],
        'author_death': row['main_author_death'],
        'source': f"https://shamela.ws/book/{row['book_id']}/{row['page'].split(', ')[0]}"
    }

In [11]:
from tqdm import tqdm

book_ids,docs,metadatas = [],[],[]

for row in tqdm(ds, total=len(ds)):

    ids = str(row['book_category'])+':'+str(row['book_id'])+':'+str(int(row['chunk_num']))
    docs.append(row['body'])

    metadata = book_meta(row)

    book_ids.append(ids)
    metadatas.append(metadata)

100%|██████████| 110984/110984 [00:34<00:00, 3252.54it/s]


In [12]:
sparse_model = SparseTextEmbedding(
    "Qdrant/bm25"
)  # BM25 TF vectors; IDF applied by Qdrant via Modifier.IDF
sparse_vecs = list(
    sparse_model.embed(ds['body_clean'])
)  # batch — one call, not per-point


In [13]:
points = []
for i in tqdm(range(len(book_ids))):
    sv = sparse_vecs[i]
    points.append(
        PointStruct(
            id=i + 1,
            vector={
                'dense': embs[i],
                "sparse": SparseVector(
                    indices=sv.indices.tolist(), values=sv.values.tolist()
                ),
            },
            payload={
                "ids": book_ids[i],
                "text": docs[i],
                **metadatas[i],
            },
        )
    )

  0%|          | 0/110984 [00:00<?, ?it/s]

100%|██████████| 110984/110984 [00:18<00:00, 5881.99it/s] 


In [14]:
for i in tqdm(range(0, len(points), 100), desc=f"Adding: sunnah"):
    client.upsert(
        collection_name="sunnah",
        wait=True,
        points=points[i : i + 100],
    )

Adding: sunnah: 100%|██████████| 1110/1110 [05:56<00:00,  3.11it/s]


# Test

In [29]:
q = 'الله اكبر الله اكبر'
em = model.encode([q], convert_to_tensor=True)[0].tolist()

In [40]:
res = client.query_points(
    collection_name="books",
    query=em,
    using='dense',
    limit=5
).points

for r in res:
    print(r.score)
    display(Markdown(r.payload['text']))
    # print(r.payload['page'])
    # print(r.payload['athar_number'])
    print('---')
res[0].payload

0.8590405


قال: اللَّه أكبر. 

---
0.74267393


فسمع رجلا يقول: الله أكبر الله أكبر. 

---
0.7361262


...

الله أكبر الله أكبر عما يقول فيه المبطلون. 

---
0.6955495


قَالَ أَحْمَدُ: يَرْوُونَ عَنِ ابْنِ عُمَرَ: يُكَبِّرُ ثَلَاثًا: اللَّهُ أَكْبَرُ اللَّهُ أَكْبَرُ اللَّهُ أَكْبَرُ، قَالَ أَحْمَدُ: كَبِّرْ تَكْبِيرَ ابْنِ مسعودٍ. 

---


{'ids': '1:20879:1754',
 'text': 'قال: اللَّه أكبر. ',
 'book_id': 20879,
 'book_name': 'الجامع لعلوم الإمام أحمد - العقيدة',
 'bood_data': 241,
 'category_name': 'العقيدة',
 'all_authors': 'أحمد بن حنبل',
 'headers': 'كتاب الصحابة/١٥٢ - مناقب على بن أبي طالب ﵁-',
 'chunk_page': '985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 999, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010',
 'author_death': 241,
 'source': 'https://shamela.ws/book/20879/985'}

In [43]:
books_ = ds.unique('book_id')

In [ ]:
ds.unique('category_name')

['العقيدة',
 'كتب السنة',
 'العلل والسؤلات الحديثية',
 'التراجم والطبقات',
 'الفقه الحنبلي',
 'الرقائق والآداب والأذكار',
 'علوم الحديث',
 'شروح الحديث']

In [34]:
ds.unique('book_name')


['السنة لعبد الله بن أحمد',
 'الحث على التجارة - من «الجامع» للخلال - ت العوضي',
 'السنة لأبي بكر بن الخلال',
 'بر الوالدين - البخاري - ت مكي',
 'المعجم الكبير للطبراني',
 'العلل ومعرفة الرجال لأحمد رواية ابنه عبد الله',
 'العقل وفضله لابن أبي الدنيا',
 'الأسامي والكنى - الإمام أحمد',
 'سؤالات أبي داود للإمام أحمد',
 'اصطناع المعروف لابن أبي الدنيا',
 'العلل ومعرفة الرجال لأحمد رواية المروذي وغيره ت صبحي السامرائي',
 'مسائل الإمام أحمد رواية ابنه عبد الله',
 'الإخلاص والنية لابن أبي الدنيا',
 'اختصاص القرآن بعوده إلى الرحيم الرحمن',
 'أصول السنة لأحمد بن حنبل',
 'قصر الأمل لابن أبي الدنيا',
 'سنن الترمذي - ت بشار',
 'مكائد الشيطان',
 'ذم الكذب - من الصمت وآداب اللسان',
 'الأربعون حديثا للآجري',
 'الإبانة الكبرى - ابن بطة',
 'الأمر بالمعروف والنهي عن المنكر- ابن أبي الدنيا',
 'الحث على التجارة - من «الجامع» للخلال - ت الحداد',
 'الزهد لابن أبي الدنيا',
 'اليقين لابن أبي الدنيا',
 'تحريم النرد والشطرنج والملاهي للآجري',
 'ذم البغى لابن أبي الدنيا',
 'ذم الملاهي لابن أبي الدنيا',
 'صفة ال

# Non_sunnah books

# fix


In [48]:
client.delete(
    collection_name="books",
    wait=True,
    points_selector=models.FilterSelector(
        filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="book_id",
                    match=models.MatchAny(any=books_)
                ),
            ],
        )
    ),
)


UpdateResult(operation_id=2213, status=<UpdateStatus.COMPLETED: 'completed'>)